# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [ ]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [ ]:
# Task — Multi-series line with highlight
# YOUR CODE HERE
# Task — Multi-series line with highlight  (SOLUTION)

# 1. Keep only Asian countries
asia = df[df['Region'] == 'Asia']
print("Asian countries:", asia['Country'].unique())

# 2. Choose the country to highlight
highlight = 'India'

# 3. Numbers for the insight title (computed from the data, not typed by hand)
h = asia[asia['Country'] == highlight].sort_values('Year')
start, end = h['CO2_Mt'].iloc[0], h['CO2_Mt'].iloc[-1]
multiple = end / start
y0, y1 = h['Year'].iloc[0], h['Year'].iloc[-1]
# Check growth multiple for every Asian country, to justify the choice of highlight
g = asia.pivot(index='Year', columns='Country', values='CO2_Mt')
print((g.iloc[-1] / g.iloc[0]).round(2).sort_values(ascending=False))
print(f"{highlight}: {start:.0f} Mt ({y0}) -> {end:.0f} Mt ({y1}) = {multiple:.1f}x")

# 4. Colour map: highlight in colour, everything else light grey
countries = asia['Country'].unique()
color_map = {c: '#2E75B6' if c == highlight else '#EEEEEE' for c in countries}

fig = px.line(asia, x='Year', y='CO2_Mt', color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''},
              template='plotly_white',
              height=450)

# Context lines: thin, no legend entry. Highlighted line: thicker.
fig.update_traces(line=dict(width=1.5), showlegend=False)
fig.update_traces(line=dict(width=3), selector=dict(name=highlight))

# 5. Draw the highlighted line last so it sits on top of the grey lines
fig.data = tuple([t for t in fig.data if t.name != highlight] +
                 [t for t in fig.data if t.name == highlight])

# 6. Direct label at the end of the highlighted line (instead of a legend)
fig.add_annotation(
    x=y1, y=end, text=highlight, showarrow=False,
    xanchor='left', xshift=6,
    font=dict(color='#2E75B6', size=13, weight='bold', family='Arial')
)

# 7. Insight title + remove clutter
fig.update_layout(
    title=f"{highlight}'s CO2 emissions grew {multiple:.1f}x between {y0} and {y1} — a steep climb in Asia",
    font=dict(size=13),
    margin=dict(r=90),
    yaxis=dict(gridcolor='#EEEEEE', gridwidth=1),
    xaxis=dict(showgrid=False),
)

fig.show()
